### 0. Problem restatement

You have:
$Number$ samples (rows)
$People = P$ participants (columns)
Each sample is a 红包分配结果: $$ x=(x_1,x_2,...,x_p) $$
with hard constraints:
$$
x_i \ge 0, \sum_{i=1}^{p} x_i=Sum
$$
#### Your goal:

Learn a probabilistic model that approximates the true distribution of allocations,
and generate new samples that:

are non-negative

sum exactly to Sum

preserve the statistical structure of the original data

### 1.Imports and configuration

In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from tqdm import tqdm

# -----------------------
# Configuration
# -----------------------
CSV_PATH = "envelopes.csv"

Sum = float(input())       # total amount (given)
People = None      # inferred from data

BATCH_SIZE = 128
EPOCHS = 100
LR = 1e-3
T = 1000           # diffusion steps

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


### 2、Normalization and Log-ratio transformation

We first convert amounts into proportions:
$$
p_i=\frac{x_i}{Sum}
$$
Now each sample lies on the probability simplex.

To apply diffusion, we map the simplex to Euclidean space using the **Additive Log-Ratio (ALR) transform**.
Choose the last person as reference (arbitrary but fixed):$$y_i=log(\frac{p_i}{p_P}) (i=1,...,P-1)$$
This gives:$$y_i \in \mathbf{R}^{P-1}$$ 

In [33]:
# -----------------------
# Load data
# -----------------------
df = pd.read_csv(CSV_PATH, header=None)
df = df.iloc[1:, 1:]
x = df.values.astype(np.float32)

People = x.shape[1]

# Convert to proportions
p = x / Sum

# Additive log-ratio transform
# y_i = log(p_i / p_last)
eps = 1e-8
y = np.log((p[:, :-1] + eps) / (p[:, -1:] + eps))

y = torch.tensor(y, dtype=torch.float32)


### 3.Time Embedding

In [34]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, t):
        t = t.float().unsqueeze(-1) / T
        return self.net(t)


### 4.MLP denoiser

In [35]:
class DenoiseMLP(nn.Module):
    def __init__(self, data_dim, hidden_dim=256):
        super().__init__()
        self.time_emb = TimeEmbedding(hidden_dim)

        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, data_dim)
        )

    def forward(self, x, t):
        t_emb = self.time_emb(t)
        x = torch.cat([x, t_emb], dim=1)
        return self.net(x)


### 5. Diffusion process (noise prediction formulation)

In [36]:
class Diffusion:
    def __init__(self, model, T):
        self.model = model
        self.T = T

        self.betas = torch.linspace(1e-4, 0.02, T).to(DEVICE)
        self.alphas = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)

    def q_sample(self, y0, t, noise):
        a_bar = self.alpha_bar[t].unsqueeze(1)
        return torch.sqrt(a_bar) * y0 + torch.sqrt(1 - a_bar) * noise

    def loss(self, y0):
        bsz = y0.size(0)
        t = torch.randint(0, self.T, (bsz,), device=DEVICE)
        noise = torch.randn_like(y0)

        y_t = self.q_sample(y0, t, noise)
        noise_pred = self.model(y_t, t)

        return nn.MSELoss()(noise_pred, noise)

    @torch.no_grad()
    def sample(self, n_samples, data_dim):
        y = torch.randn(n_samples, data_dim).to(DEVICE)

        for t in tqdm(reversed(range(self.T)), desc="Sampling"):
            t_batch = torch.full((n_samples,), t, device=DEVICE)

            beta = self.betas[t]
            alpha = self.alphas[t]
            a_bar = self.alpha_bar[t]

            eps = self.model(y, t_batch)
            mean = (1 / torch.sqrt(alpha)) * (
                y - beta / torch.sqrt(1 - a_bar) * eps
            )

            if t > 0:
                y = mean + torch.sqrt(beta) * torch.randn_like(y)
            else:
                y = mean

        return y


### 6.Training

In [37]:
dataset = torch.utils.data.TensorDataset(y)
loader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True
)

model = DenoiseMLP(data_dim=People - 1).to(DEVICE)
diffusion = Diffusion(model, T)
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    total_loss = 0.0
    for (y0,) in loader:
        y0 = y0.to(DEVICE)

        loss = diffusion.loss(y0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:03d} | Loss: {total_loss / len(loader):.6f}")


Epoch 001 | Loss: 0.972805
Epoch 002 | Loss: 0.932613
Epoch 003 | Loss: 0.928450
Epoch 004 | Loss: 0.846011
Epoch 005 | Loss: 0.743362
Epoch 006 | Loss: 0.799287
Epoch 007 | Loss: 0.784911
Epoch 008 | Loss: 0.739918
Epoch 009 | Loss: 0.684479
Epoch 010 | Loss: 0.642693
Epoch 011 | Loss: 0.661968
Epoch 012 | Loss: 0.623662
Epoch 013 | Loss: 0.561862
Epoch 014 | Loss: 0.567674
Epoch 015 | Loss: 0.528946
Epoch 016 | Loss: 0.432368
Epoch 017 | Loss: 0.452834
Epoch 018 | Loss: 0.390381
Epoch 019 | Loss: 0.478719
Epoch 020 | Loss: 0.439409
Epoch 021 | Loss: 0.391380
Epoch 022 | Loss: 0.404212
Epoch 023 | Loss: 0.386985
Epoch 024 | Loss: 0.353354
Epoch 025 | Loss: 0.360956
Epoch 026 | Loss: 0.371454
Epoch 027 | Loss: 0.425947
Epoch 028 | Loss: 0.395651
Epoch 029 | Loss: 0.315988
Epoch 030 | Loss: 0.368380
Epoch 031 | Loss: 0.325461
Epoch 032 | Loss: 0.330199
Epoch 033 | Loss: 0.317718
Epoch 034 | Loss: 0.374111
Epoch 035 | Loss: 0.362051
Epoch 036 | Loss: 0.389106
Epoch 037 | Loss: 0.322420
E

### 7.Sampling and inverse transform (guaranteed sum)

In [39]:
# -----------------------
# Generate new samples
# -----------------------
n_samples = 1000
y_gen = diffusion.sample(n_samples, People - 1)

# Inverse ALR transform
y_gen = y_gen.cpu().numpy()
exp_y = np.exp(y_gen)

p_last = np.ones((n_samples, 1))
p_all = np.concatenate([exp_y, p_last], axis=1)
p_all = p_all / p_all.sum(axis=1, keepdims=True)

def round_to_cents_preserve_sum(x, decimals=2):
    """
    x: numpy array of shape (People,)
    returns: rounded array with fixed decimals and preserved sum
    """
    scale = 10 ** decimals
    c = x * scale

    floor_c = np.floor(c)
    remainder = c - floor_c

    deficit = int(round(scale * x.sum() - floor_c.sum()))

    # indices of largest remainders
    idx = np.argsort(remainder)[::-1][:deficit]
    floor_c[idx] += 1

    return floor_c / scale


# Recover amounts
x_gen = np.vstack([
    round_to_cents_preserve_sum(row, decimals=2)
    for row in x_gen
])


# Save
pd.DataFrame(x_gen).to_csv(
    "generated_samples.csv",
    index=False,
    header=False
)

print("Generated samples saved. Total sum per row:",
      np.unique(x_gen.sum(axis=1)))


Sampling: 1000it [00:02, 404.80it/s]


Generated samples saved. Total sum per row: [20. 20. 20.]
